This notebook executes against local measurement inputs that are deliberately not in the
repository: the generated expectation table under `~/.sigwood/bench/protocol/` and the Zeek
connection archive under `~/.sigwood/exports/zeek/`. Without them the table-loading cell stops.
Every printed cell is aggregate only; addresses in fixtures are RFC 5737 documentation space.

In [ ]:
import time
import warnings
from sklearn import __version__ as SKLEARN_VERSION
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPRegressor

def fit_leg_c_class(features: list[dict], *, class_floor: int, surviving_floor: int, fit_cap: int, fallback_cap: float, topology: tuple[int, ...], seed: int, early_stopping: bool, validation_fraction: float, max_iter: int, batch_size: int, n_iter_no_change: int = 10, tol: float = 1e-4) -> dict:
    if len(features) < class_floor:
        return {'status': 'below_class_floor', 'rows': len(features)}
    fit_rows = seeded_fit_rows(features, cap=fit_cap, seed=seed)
    fit_numeric = np.stack([row['numeric'] for row in fit_rows])
    fit_binary = np.stack([row['binary'] for row in fit_rows])
    scaler = fit_leg_c_scaler(fit_numeric, fallback_cap=fallback_cap)
    surviving = int(scaler['retained'].sum()) + fit_binary.shape[1]
    if surviving < surviving_floor:
        return {'status': 'below_feature_floor', 'rows': len(features), 'surviving_features': surviving, 'constant_names': scaler['constant_names']}
    x_fit = apply_leg_c_scaler(fit_numeric, fit_binary, scaler)
    model = MLPRegressor(hidden_layer_sizes=topology, random_state=seed, early_stopping=early_stopping, validation_fraction=validation_fraction, max_iter=max_iter, batch_size=batch_size, n_iter_no_change=n_iter_no_change, tol=tol)
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always', ConvergenceWarning)
            model.fit(x_fit, x_fit)
    except Exception as error:
        return {'status': 'failed', 'rows': len(features), 'error_type': type(error).__name__}
    if any(issubclass(item.category, ConvergenceWarning) for item in caught):
        return {'status': 'not_converged', 'rows': len(features), 'iterations': getattr(model, 'n_iter_', None)}
    return {'status': 'modeled', 'rows': len(features), 'fit_rows': len(fit_rows), 'surviving_features': surviving, 'iterations': model.n_iter_, 'loss': float(model.loss_), 'scaler': scaler, 'model': model}

def score_leg_c_class(features: list[dict], fitted: dict, *, score_batch_size: int) -> dict:
    if fitted.get('status') != 'modeled':
        return {'status': fitted.get('status', 'failed'), 'errors': np.asarray([], dtype=np.float64)}
    numeric = np.stack([row['numeric'] for row in features])
    binary = np.stack([row['binary'] for row in features])
    blind_spot = leg_c_constant_blind_spot(numeric, fitted['scaler'])
    scoreable = np.flatnonzero(~blind_spot)
    errors = np.full(len(features), np.nan, dtype=np.float64)
    worst = np.full(len(features), '', dtype=object)
    feature_names = tuple(name for name, keep in zip(LEG_C_NUMERIC_NAMES, fitted['scaler']['retained'], strict=True) if keep) + LEG_C_BINARY_NAMES
    for start in range(0, len(scoreable), score_batch_size):
        indexes = scoreable[start:start + score_batch_size]
        x_score = apply_leg_c_scaler(numeric[indexes], binary[indexes], fitted['scaler'])
        residual = np.square(x_score - fitted['model'].predict(x_score))
        errors[indexes] = residual.mean(axis=1)
        worst[indexes] = [feature_names[int(index)] for index in residual.argmax(axis=1)]
    return {'status': 'scored', 'errors': errors, 'worst_feature': worst, 'constant_blind_spot': blind_spot}

def score_robust_z_baseline(features: list[dict], scaler: dict) -> dict:
    numeric = np.stack([row['numeric'] for row in features])
    blind_spot = leg_c_constant_blind_spot(numeric, scaler)
    scores = np.full(len(features), np.nan, dtype=np.float64)
    if scaler['retained'].any():
        z = np.abs((numeric[~blind_spot][:, scaler['retained']] - scaler['centers'][scaler['retained']]) / scaler['scales'][scaler['retained']])
        scores[~blind_spot] = z.max(axis=1)
    return {'status': 'scored', 'scores': scores, 'constant_blind_spot': blind_spot}

def robust_score_bar(scores: np.ndarray, *, population_floor: int, multiplier: float, cliff_gap: float) -> dict:
    finite = np.sort(np.asarray(scores, dtype=np.float64)[np.isfinite(scores)])
    if len(finite) < population_floor:
        return {'status': 'below_population_floor', 'population': len(finite)}
    center = float(np.median(finite))
    q25, q75 = np.percentile(finite, (25, 75))
    scale, scale_kind = float(q75 - q25), 'iqr'
    if scale == 0:
        mad = float(np.median(np.abs(finite - center)))
        span = float(np.ptp(finite))
        scale, scale_kind = (mad, 'mad') if mad > 0 else ((span, 'range') if span > 0 else (1.0, 'unit'))
    threshold = center + multiplier * scale
    split = int(np.searchsorted(finite, threshold, side='right'))
    if split == len(finite):
        return {'status': 'no_tail', 'population': len(finite), 'center': center, 'scale': scale, 'scale_kind': scale_kind, 'threshold': threshold, 'tail': 0}
    lower = finite[split - 1] if split else center
    observed_gap = float((finite[split] - lower) / scale)
    if observed_gap < cliff_gap:
        return {'status': 'no_cliff', 'population': len(finite), 'center': center, 'scale': scale, 'scale_kind': scale_kind, 'threshold': threshold, 'tail': len(finite) - split, 'observed_cliff_gap': observed_gap}
    return {'status': 'ready', 'population': len(finite), 'center': center, 'scale': scale, 'scale_kind': scale_kind, 'threshold': threshold, 'tail': len(finite) - split, 'observed_cliff_gap': observed_gap}

def aggregate_pair_scores(pair_keys: list[tuple], scores: np.ndarray, *, threshold: float, share_floor: float) -> list[dict]:
    aggregates = defaultdict(lambda: {'rows': 0, 'above': 0, 'worst': -math.inf})
    for pair_key, score in zip(pair_keys, scores, strict=True):
        if not math.isfinite(float(score)):
            continue
        aggregate = aggregates[pair_key]
        aggregate['rows'] += 1
        aggregate['above'] += int(score > threshold)
        aggregate['worst'] = max(aggregate['worst'], float(score))
    return [
        {'pair_key': pair_key, **aggregate, 'share': aggregate['above'] / aggregate['rows']}
        for pair_key, aggregate in sorted(aggregates.items())
        if aggregate['above'] / aggregate['rows'] >= share_floor
    ]

def folded_pair_finding_count(findings: list[dict], *, fold_count: int = 4) -> int:
    groups = Counter((item['pair_key'][4], item['pair_key'][2], item['pair_key'][3]) for item in findings)
    return sum(1 if members >= fold_count else members for members in groups.values())

LEG_C_MODEL_CANDIDATES = {
    'compact-16-8-16': (16, 8, 16),
    'medium-32-16-32': (32, 16, 32),
}
LEG_C_CANDIDATE_CLASS_FLOOR = 500
LEG_C_CANDIDATE_SURVIVING_FLOOR = 10
LEG_C_CANDIDATE_FALLBACK_CAP = 10.0
LEG_C_CANDIDATE_MAX_ITER = 500
LEG_C_CANDIDATE_BATCH_SIZE = 512
LEG_C_CANDIDATE_VALIDATION_FRACTION = 0.1

def run_leg_c_model_grid(samples: dict) -> dict:
    result = {'schema_version': 1, 'sklearn_version': SKLEARN_VERSION, 'parameters': {'class_floor': LEG_C_CANDIDATE_CLASS_FLOOR, 'surviving_floor': LEG_C_CANDIDATE_SURVIVING_FLOOR, 'fit_cap': LEG_C_CANDIDATE_MAX_FIT_CAP, 'fallback_cap': LEG_C_CANDIDATE_FALLBACK_CAP, 'seed': LEG_C_SAMPLE_SEED, 'early_stopping': True, 'validation_fraction': LEG_C_CANDIDATE_VALIDATION_FRACTION, 'max_iter': LEG_C_CANDIDATE_MAX_ITER, 'batch_size': LEG_C_CANDIDATE_BATCH_SIZE}, 'candidates': []}
    for label, topology in LEG_C_MODEL_CANDIDATES.items():
        candidate_started = time.perf_counter()
        classes = []
        for class_key, rows in sorted(samples.items()):
            class_started = time.perf_counter()
            fitted = fit_leg_c_class(rows, class_floor=LEG_C_CANDIDATE_CLASS_FLOOR, surviving_floor=LEG_C_CANDIDATE_SURVIVING_FLOOR, fit_cap=LEG_C_CANDIDATE_MAX_FIT_CAP, fallback_cap=LEG_C_CANDIDATE_FALLBACK_CAP, topology=topology, seed=LEG_C_SAMPLE_SEED, early_stopping=True, validation_fraction=LEG_C_CANDIDATE_VALIDATION_FRACTION, max_iter=LEG_C_CANDIDATE_MAX_ITER, batch_size=LEG_C_CANDIDATE_BATCH_SIZE)
            record = {'services': list(class_key[0]), 'proto': class_key[1], 'rows': len(rows), 'status': fitted['status'], 'elapsed_seconds': time.perf_counter() - class_started}
            for key in ('fit_rows', 'surviving_features', 'iterations', 'loss', 'error_type'):
                if key in fitted:
                    record[key] = fitted[key]
            if fitted['status'] == 'modeled':
                scored = score_leg_c_class(rows, fitted, score_batch_size=4096)
                finite_errors = scored['errors'][np.isfinite(scored['errors'])]
                record['fit_error'] = {'median': float(np.median(finite_errors)), 'q75': float(np.percentile(finite_errors, 75)), 'q99': float(np.percentile(finite_errors, 99)), 'max': float(finite_errors.max()), 'constant_blind_spots': int(scored['constant_blind_spot'].sum())}
                record['constant_numeric'] = list(fitted['scaler']['constant_names'])
                record['fallback_kinds'] = list(fitted['scaler']['scale_kind'])
            classes.append(record)
        result['candidates'].append({'label': label, 'topology': topology, 'elapsed_seconds': time.perf_counter() - candidate_started, 'classes': classes})
    return result

def write_leg_c_model_grid(result: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(result, sort_keys=True, indent=2) + '\n', encoding='utf-8')

# Executable fixtures follow the feature/scaling cell that defines their inputs.

# protocol detector calibration

U-2026-09-08-01 working notebook. Measured corpora stay outside the repository; rendered examples use reserved identities and aggregate counts only. Thresholds remain unset until development-window calibration.

In [ ]:
from __future__ import annotations

import gzip
import json
import math
import tempfile
import sqlite3
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable

EXPECTATION_TABLE = Path.home() / '.sigwood/bench/protocol/services-v8.2.1-2026-09-09-run1'
DEVELOPMENT_WEEKS = ('2026-05-18--05-24', '2026-06-08--06-14', '2026-06-29--07-05', '2026-07-20--07-26')
INITIAL_HOLDBACK_WEEKS = ('2026-08-03--08-09', '2026-08-17--08-23')
REDESIGN_RESERVE_WEEKS = ('2026-08-24--08-30', '2026-08-31--09-06')
assert set(DEVELOPMENT_WEEKS).isdisjoint(INITIAL_HOLDBACK_WEEKS + REDESIGN_RESERVE_WEEKS)

In [ ]:
LEG_B_REFERENCE_FLOORS = (100, 1000, 10000)
LEG_B_LABELED_SHARES = (0.90, 0.99, 0.999)

def _valid_identity(row: dict):
    try:
        ts = float(row['ts'])
        port = int(row['id.resp_p'])
        src, dst, proto = str(row['id.orig_h']), str(row['id.resp_h']), str(row['proto'])
    except (KeyError, TypeError, ValueError):
        return None
    if not math.isfinite(ts) or not src or not dst or proto not in {'tcp', 'udp'} or not 0 < port <= 65535:
        return None
    return ts, src, dst, port, proto

def scan_leg_b_week(paths: Iterable[Path], week: str, reference_floors=LEG_B_REFERENCE_FLOORS, labeled_shares=LEG_B_LABELED_SHARES) -> dict:
    allowed_dates = _week_dates(week)
    counters = Counter()
    with tempfile.TemporaryDirectory(prefix='sigwood-protocol-legb-') as directory:
        database = sqlite3.connect(Path(directory) / 'ordered.sqlite3')
        database.execute('PRAGMA journal_mode=OFF')
        database.execute('PRAGMA synchronous=OFF')
        database.execute('PRAGMA temp_store=FILE')
        database.execute('CREATE TABLE rows(ts REAL, src TEXT, dst TEXT, port INTEGER, proto TEXT, quality INTEGER, candidate INTEGER, labeled INTEGER)')
        pending = []
        for path in paths:
            for row in iter_zeek_ndjson(path):
                counters['input_rows'] += 1
                identity = _valid_identity(row)
                if identity is None:
                    counters['invalid_identity'] += 1
                    continue
                ts, src, dst, port, proto = identity
                if datetime.fromtimestamp(ts, timezone.utc).date().isoformat() not in allowed_dates:
                    counters['outside_window'] += 1
                    continue
                services = parse_services(row.get('service'))
                supplied_unlabeled = row.get('_source_has_service') is True and not services.confirmed and not services.removed
                quality, reason = leg_b_quality(row)
                counters[f'quality_{reason}'] += 1
                if supplied_unlabeled:
                    counters['supplied_unlabeled'] += 1
                    counters['eligible_unlabeled' if quality else f'unlabeled_loss_{reason}'] += 1
                pending.append((ts, src, dst, port, proto, int(quality), int(supplied_unlabeled and quality), int(bool(services.confirmed))))
                if len(pending) >= 10000:
                    database.executemany('INSERT INTO rows VALUES (?,?,?,?,?,?,?,?)', pending)
                    pending.clear()
        if pending:
            database.executemany('INSERT INTO rows VALUES (?,?,?,?,?,?,?,?)', pending)
        database.commit()
        database.execute('CREATE INDEX rows_ts ON rows(ts)')
        database.execute('CREATE TABLE seen(src TEXT, dst TEXT, port INTEGER, proto TEXT, PRIMARY KEY(src,dst,port,proto)) WITHOUT ROWID')
        database.commit()

        reference = defaultdict(lambda: Counter(total=0, labeled=0))
        seed_facts = []
        current_ts, batch = None, []
        def consume_batch(rows):
            candidates = {}
            for _ts, src, dst, port, proto, quality, candidate, labeled in rows:
                identity = (src, dst, port, proto)
                if candidate:
                    candidates.setdefault(identity, (proto, port))
            for identity, port_key in candidates.items():
                prior = database.execute('SELECT 1 FROM seen WHERE src=? AND dst=? AND port=? AND proto=?', identity).fetchone()
                if prior is None:
                    counts = reference[port_key]
                    share = counts['labeled'] / counts['total'] if counts['total'] else 0.0
                    seed_facts.append((counts['total'], share))
            database.executemany('INSERT OR IGNORE INTO seen VALUES (?,?,?,?)', ((src, dst, port, proto) for _ts, src, dst, port, proto, _quality, _candidate, _labeled in rows))
            for _ts, _src, _dst, port, proto, quality, _candidate, labeled in rows:
                if quality:
                    reference[(proto, port)]['total'] += 1
                    reference[(proto, port)]['labeled'] += labeled

        for row in database.execute('SELECT ts,src,dst,port,proto,quality,candidate,labeled FROM rows ORDER BY ts'):
            if current_ts is None or row[0] == current_ts:
                current_ts = row[0]
                batch.append(row)
            else:
                consume_batch(batch)
                current_ts, batch = row[0], [row]
        if batch:
            consume_batch(batch)
        database.commit()
        database.close()

    grid = []
    for floor in reference_floors:
        for share in labeled_shares:
            grid.append({'reference_floor': floor, 'labeled_share': share, 'surfaced_seed_events': sum(total >= floor and labeled >= share for total, labeled in seed_facts)})
    return {'schema_version': 1, 'week': week, 'grid': grid, 'new_eligible_seed_events': len(seed_facts), 'counts': dict(counters)}

def write_leg_b_development(root: Path = EXPECTATION_TABLE.parent) -> list[Path]:
    outputs = []
    grid_tag = 'r100-1000-10000_s900-990-999-v2-reasons'
    for week in DEVELOPMENT_WEEKS:
        paths = conn_files_for_weeks(ESTATE_ROOT, (week,))
        result = scan_leg_b_week(paths, week)
        output = root / f"leg-b-dev-{week.replace('--', '_')}_{grid_tag}.json"
        output.write_text(json.dumps(result, sort_keys=True, indent=2) + '\n', encoding='utf-8')
        outputs.append(output)
    return outputs

LEG_B_FROZEN_REFERENCE_FLOOR = 100
LEG_B_FROZEN_LABELED_SHARE = 0.999

def write_leg_b_initial_heldback(root: Path = EXPECTATION_TABLE.parent) -> list[Path]:
    outputs = []
    for week in INITIAL_HOLDBACK_WEEKS:
        paths = conn_files_for_weeks(ESTATE_ROOT, (week,))
        result = scan_leg_b_week(paths, week, (LEG_B_FROZEN_REFERENCE_FLOOR,), (LEG_B_FROZEN_LABELED_SHARE,))
        output = root / f"leg-b-heldback-{week.replace('--', '_')}_r100-s999-frozen-v1.json"
        output.write_text(json.dumps(result, sort_keys=True, indent=2) + '\n', encoding='utf-8')
        outputs.append(output)
    return outputs

print('leg B development grid harness:', len(LEG_B_REFERENCE_FLOORS) * len(LEG_B_LABELED_SHARES), 'parameter pairs; no scan executed')
# Director-owned development run: LEG_B_DEVELOPMENT_ARTIFACTS = write_leg_b_development()
# Frozen held-back run, only after director verdict: LEG_B_HELDBACK_ARTIFACTS = write_leg_b_initial_heldback()

In [ ]:
OBSERVED_FIELDS = ('service', 'history', 'missed_bytes', 'orig_pkts', 'resp_pkts', 'orig_ip_bytes', 'resp_ip_bytes')
OBSERVATION_FLAGS = {name: f'_source_has_{name}' for name in OBSERVED_FIELDS}

@dataclass(frozen=True)
class ServiceSet:
    confirmed: tuple[str, ...]
    removed: tuple[str, ...]

def parse_services(raw: object) -> ServiceSet:
    if raw is None or (isinstance(raw, float) and math.isnan(raw)):
        return ServiceSet((), ())
    confirmed, removed = set(), set()
    for token in str(raw).split(','):
        token = token.strip().lower()
        if not token or token == '-':
            continue
        if token.startswith('-') and len(token) > 1:
            removed.add(token[1:])
        else:
            confirmed.add(token)
    return ServiceSet(tuple(sorted(confirmed)), tuple(sorted(removed)))

def _metadata(comment: str) -> dict[str, str]:
    result = {}
    for field in comment.strip().split():
        if '=' in field:
            key, value = field.split('=', 1)
            result[key] = value
    return result

def load_expectations(path: Path) -> dict:
    ports = defaultdict(lambda: {'registered': set(), 'conventional': set()})
    flags = defaultdict(dict)
    provenance = {}
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if line.startswith('# zeek_tag=') or line.startswith('# generated_on=') or line.startswith('# generator='):
            key, value = line[2:].split('=', 1)
            provenance[key] = value
        elif line.startswith('# flag '):
            meta = _metadata(line.removeprefix('# flag '))
            flags[meta['service']][meta['kind']] = tuple(filter(None, meta.get('values', '').split(',')))
        elif line and not line.startswith('#'):
            body, comment = line.split('#', 1)
            service, port_proto = body.split()
            meta = _metadata(comment)
            port, proto = port_proto.split('/')
            ports[service][meta['tier']].add((int(port), proto))
    return {'ports': dict(ports), 'flags': dict(flags), 'provenance': provenance}

EXPECTATIONS = load_expectations(EXPECTATION_TABLE)
assert EXPECTATIONS['provenance']['zeek_tag'] == 'v8.2.1'
assert (994, 'tcp') not in EXPECTATIONS['ports']['ssl']['registered']
print('loaded', sum(len(tier) for row in EXPECTATIONS['ports'].values() for tier in row.values()), 'expectation rows')

In [ ]:
def classify_labels(services: ServiceSet, port: int, proto: str, table: dict = EXPECTATIONS) -> dict[str, str]:
    explicit = {}
    for label in services.confirmed:
        record = table['ports'].get(label)
        if record and (port, proto) in record['registered']:
            explicit[label] = 'registered'
        elif record and (port, proto) in record['conventional']:
            explicit[label] = 'conventional-only'
    outcomes = dict(explicit)
    for label in services.confirmed:
        if label in outcomes:
            continue
        record = table['ports'].get(label)
        flags = table['flags'].get(label, {})
        carriers = flags.get('dependent_on', ())
        if any(carrier in services.confirmed and explicit.get(carrier) in {'registered', 'conventional-only'} for carrier in carriers):
            outcomes[label] = 'dependent-exempt'
        elif record is None or 'negotiated' in flags or 'encapsulation' in flags:
            outcomes[label] = 'unavailable'
        elif carriers and not any(carrier in services.confirmed for carrier in carriers) and any(
            (port, proto) in table['ports'].get(carrier, {}).get(tier, set())
            for carrier in carriers for tier in ('registered', 'conventional')
        ):
            outcomes[label] = 'ambiguous'
        else:
            outcomes[label] = 'off-expectation'
    return outcomes

def pair_verdict(outcomes: dict[str, str]) -> tuple[str, tuple[str, ...]]:
    off = tuple(sorted(label for label, outcome in outcomes.items() if outcome == 'off-expectation'))
    conventional = tuple(sorted(label for label, outcome in outcomes.items() if outcome == 'conventional-only'))
    if off:
        return 'mismatch', off + conventional
    if conventional:
        return 'conventional', conventional
    if any(value == 'ambiguous' for value in outcomes.values()):
        return 'ambiguous', ()
    if any(value == 'unavailable' for value in outcomes.values()):
        return 'unavailable', ()
    return 'conformant', ()

PINNED_CASES = (
    ('ssh', 22, 'tcp', {'ssh': 'registered'}, ('conformant', ())),
    ('ssh', 2222, 'tcp', {'ssh': 'off-expectation'}, ('mismatch', ('ssh',))),
    ('ssh', 443, 'tcp', {'ssh': 'off-expectation'}, ('mismatch', ('ssh',))),
    ('ssh,ssl', 443, 'tcp', {'ssh': 'off-expectation', 'ssl': 'registered'}, ('mismatch', ('ssh',))),
    ('ssh,http', 80, 'tcp', {'http': 'registered', 'ssh': 'off-expectation'}, ('mismatch', ('ssh',))),
    ('smtp,ssl', 25, 'tcp', {'smtp': 'registered', 'ssl': 'dependent-exempt'}, ('conformant', ())),
    ('http,ssl', 80, 'tcp', {'http': 'registered', 'ssl': 'dependent-exempt'}, ('conformant', ())),
    ('quic,ssl', 443, 'udp', {'quic': 'registered', 'ssl': 'dependent-exempt'}, ('conformant', ())),
    ('ssl', 8443, 'tcp', {'ssl': 'conventional-only'}, ('conventional', ('ssl',))),
    ('ssl', 25, 'tcp', {'ssl': 'ambiguous'}, ('ambiguous', ())),
    ('ssl', 2222, 'tcp', {'ssl': 'off-expectation'}, ('mismatch', ('ssl',))),
    ('dce_rpc', 49152, 'tcp', {'dce_rpc': 'unavailable'}, ('unavailable', ())),
    ('ayiya', 443, 'udp', {'ayiya': 'unavailable'}, ('unavailable', ())),
    ('zzz', 1, 'tcp', {'zzz': 'unavailable'}, ('unavailable', ())),
)
for raw, port, proto, expected_outcomes, expected_verdict in PINNED_CASES:
    actual_outcomes = classify_labels(parse_services(raw), port, proto)
    assert actual_outcomes == expected_outcomes, (raw, actual_outcomes)
    assert pair_verdict(actual_outcomes) == expected_verdict, raw
print(f'pinned truth table: {len(PINNED_CASES)}/{len(PINNED_CASES)} passed')

In [ ]:
def _canonical_row(row: dict, observed: set[str], source: str) -> dict:
    result = dict(row)
    result['_source_file'] = source
    for field in OBSERVED_FIELDS:
        result.setdefault(field, None)
        result[OBSERVATION_FLAGS[field]] = field in observed
    return result

def _text_lines(path: Path):
    opener = gzip.open if path.suffix == '.gz' else open
    with opener(path, 'rt', encoding='utf-8') as handle:
        yield from handle

def read_zeek_tsv(path: Path) -> list[dict]:
    fields = None
    rows = []
    for raw_line in _text_lines(path):
        line = raw_line.rstrip('\n')
        if line.startswith('#fields'):
            fields = line.split('\t')[1:]
        elif line and not line.startswith('#'):
            if fields is None:
                raise ValueError(f'{path}: data before #fields')
            values = line.split('\t')
            if len(values) != len(fields):
                raise ValueError(f'{path}: row width differs from #fields')
            row = {key: (None if value in {'-', '(empty)'} else value) for key, value in zip(fields, values)}
            rows.append(_canonical_row(row, set(fields), path.name))
    if fields is None:
        raise ValueError(f'{path}: missing #fields')
    return rows

def iter_zeek_ndjson(path: Path):
    observed = set()
    for line in _text_lines(path):
        if not line.strip():
            continue
        row = json.loads(line)
        if not isinstance(row, dict):
            raise ValueError(f'{path}: every NDJSON record must be an object')
        observed.update(field for field in OBSERVED_FIELDS if field in row)
    for line in _text_lines(path):
        if not line.strip():
            continue
        row = json.loads(line)
        if not isinstance(row, dict):
            raise ValueError(f'{path}: every NDJSON record must be an object')
        yield _canonical_row(row, observed, path.name)

def read_zeek_ndjson(path: Path) -> list[dict]:
    return list(iter_zeek_ndjson(path))

def assert_observation_fixture(reader, with_path: Path, without_path: Path) -> None:
    for paths in ((with_path, without_path), (without_path, with_path)):
        rows = [row for path in paths for row in reader(path)]
        by_source = defaultdict(list)
        for row in rows:
            by_source[row['_source_file']].append(row)
        for row in by_source[with_path.name]:
            assert all(row[OBSERVATION_FLAGS[field]] is True for field in OBSERVED_FIELDS)
        assert any(row['service'] is None for row in by_source[with_path.name])
        for row in by_source[without_path.name]:
            assert all(row[OBSERVATION_FLAGS[field]] is False for field in OBSERVED_FIELDS)
            assert all(row[field] is None for field in OBSERVED_FIELDS)

with tempfile.TemporaryDirectory() as directory:
    root = Path(directory)
    fields = ('ts',) + OBSERVED_FIELDS
    tsv_with = root / 'with-fields.tsv'
    tsv_without = root / 'without-fields.tsv'
    tsv_with.write_text('#separator \x09\n#fields\t' + '\t'.join(fields) + '\n1.0\t-\tShDd\t0\t1\t1\t60\t60\n', encoding='utf-8')
    tsv_without.write_text('#separator \x09\n#fields\tts\tuid\n2.0\tC1\n', encoding='utf-8')
    assert_observation_fixture(read_zeek_tsv, tsv_with, tsv_without)
    ndjson_with = root / 'with-fields.ndjson'
    ndjson_without = root / 'without-fields.ndjson'
    ndjson_with.write_text(json.dumps({'ts': 1.0, **{field: None for field in OBSERVED_FIELDS}}) + '\n', encoding='utf-8')
    ndjson_without.write_text(json.dumps({'ts': 2.0, 'uid': 'C1'}) + '\n', encoding='utf-8')
    assert_observation_fixture(read_zeek_ndjson, ndjson_with, ndjson_without)
print('observation flags: TSV and NDJSON two-file fixtures passed in both concatenation orders')

In [ ]:
def _week_dates(spec: str) -> set[str]:
    start_text, end_text = spec.split('--')
    start = datetime.fromisoformat(start_text).date()
    end = datetime.fromisoformat(end_text if len(end_text) == 10 else f'{start_text[:5]}{end_text}').date()
    return {(start.fromordinal(day)).isoformat() for day in range(start.toordinal(), end.toordinal() + 1)}

def conn_files_for_weeks(root: Path, weeks: Iterable[str]) -> list[Path]:
    dates = set().union(*(_week_dates(week) for week in weeks))
    paths = []
    for day in sorted(dates):
        day_root = root / day
        for path in sorted(day_root.glob('conn*.log*')):
            if path.is_file() and not path.name.startswith('conn-summary'):
                paths.append(path)
    return paths

def scan_service_shapes(paths: Iterable[Path]) -> dict:
    shapes = Counter()
    days = defaultdict(set)
    source_flags = Counter()
    evaluation_losses = Counter()
    rows = 0
    for path in paths:
        for row in iter_zeek_ndjson(path):
            rows += 1
            source_flags['service_supplied' if row['_source_has_service'] else 'service_not_supplied'] += 1
            if row.get('_source_has_service') is not True:
                evaluation_losses['service_unavailable'] += 1
                continue
            services = parse_services(row.get('service'))
            if not services.confirmed:
                continue
            resp_endpoint = (int(row.get('id.resp_p', 0)), str(row.get('proto')))
            orig_endpoint = (int(row.get('id.orig_p', 0)), str(row.get('proto')))
            role_inverted = '^' in str(row.get('history') or '') and any(
                orig_endpoint in (EXPECTATIONS['ports'].get(label, {}).get('registered', set()) | EXPECTATIONS['ports'].get(label, {}).get('conventional', set()))
                and resp_endpoint not in (EXPECTATIONS['ports'].get(label, {}).get('registered', set()) | EXPECTATIONS['ports'].get(label, {}).get('conventional', set()))
                for label in services.confirmed
            )
            if role_inverted:
                evaluation_losses['role_inverted'] += 1
                continue
            key = (services.confirmed, services.removed, str(row.get('proto')), int(row.get('id.resp_p', 0)))
            shapes[key] += 1
            days[key].add(path.parent.name)
    denominators = Counter()
    for (confirmed, _removed, proto, _port), count in shapes.items():
        denominators[(confirmed, proto)] += count
    records = []
    for key, count in sorted(shapes.items()):
        confirmed, removed, proto, port = key
        outcomes = classify_labels(ServiceSet(confirmed, removed), port, proto)
        verdict, mismatched = pair_verdict(outcomes)
        records.append({'services': confirmed, 'removed_services': removed, 'proto': proto, 'port': port, 'conns': count, 'share': count / denominators[(confirmed, proto)], 'days_present': len(days[key]), 'label_outcomes': outcomes, 'verdict': verdict, 'mismatched_services': mismatched})
    return {'rows': rows, 'source_flags': dict(source_flags), 'evaluation_losses': dict(evaluation_losses), 'records': records}

def write_service_shape_artifact(result: dict, path: Path) -> None:
    payload = {'schema_version': 1, 'zeek_tag': EXPECTATIONS['provenance']['zeek_tag'], 'development_weeks': DEVELOPMENT_WEEKS, **result}
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, sort_keys=True, indent=2) + '\n', encoding='utf-8')

ESTATE_ROOT = Path.home() / '.sigwood/exports/zeek'
DEVELOPMENT_CONN_FILES = conn_files_for_weeks(ESTATE_ROOT, DEVELOPMENT_WEEKS)
assert DEVELOPMENT_CONN_FILES
assert all(path.parent.name in set().union(*(_week_dates(week) for week in DEVELOPMENT_WEEKS)) for path in DEVELOPMENT_CONN_FILES)
assert not any(path.parent.name in set().union(*(_week_dates(week) for week in INITIAL_HOLDBACK_WEEKS + REDESIGN_RESERVE_WEEKS)) for path in DEVELOPMENT_CONN_FILES)
DEVELOPMENT_SHAPE_ARTIFACT = EXPECTATION_TABLE.parent / 'leg-a-dev-4x7d-2026-05-18_2026-06-08_2026-06-29_2026-07-20-v8.2.1-r3-structural-role.json'
print('development harness:', len(DEVELOPMENT_CONN_FILES), 'conn files; held-back paths excluded')
# Director-owned scale run: result = scan_service_shapes(DEVELOPMENT_CONN_FILES)
# write_service_shape_artifact(result, DEVELOPMENT_SHAPE_ARTIFACT)

In [ ]:
BAD_HISTORY = frozenset('gGcCxX')
REQUIRED_HISTORY = frozenset('ShDd')

def leg_b_quality(row: dict) -> tuple[bool, str]:
    required = ('service', 'history', 'missed_bytes')
    if any(row.get(OBSERVATION_FLAGS[field]) is not True for field in required):
        return False, 'required-input-not-supplied'
    services = parse_services(row.get('service'))
    if row.get('proto') != 'tcp':
        return False, 'non-tcp'
    if services.removed:
        return False, 'removed-analyzer'
    history = set(row.get('history') or '')
    if '^' in history and not services.confirmed:
        return False, 'direction-flipped-unlabeled'
    if row.get('conn_state') != 'SF':
        return False, 'not-sf'
    if float(row.get('orig_bytes') or 0) <= 0 or float(row.get('resp_bytes') or 0) <= 0:
        return False, 'payload-not-both-positive'
    if float(row.get('missed_bytes') or 0) != 0:
        return False, 'missed-bytes'
    if not REQUIRED_HISTORY.issubset(history):
        return False, 'history-missing-required'
    if history & BAD_HISTORY:
        return False, 'history-capture-quality'
    return True, 'eligible'

def leg_b_walk(rows: Iterable[dict], reference_floor: int, labeled_share: float) -> list[dict]:
    pairs_seen = set()
    reference = defaultdict(lambda: Counter(total=0, labeled=0))
    surfaced = []
    by_time = defaultdict(list)
    for row in rows:
        by_time[float(row['ts'])].append(row)
    for timestamp in sorted(by_time):
        batch = by_time[timestamp]
        seeds = {}
        for row in batch:
            identity = (row['id.orig_h'], row['id.resp_h'], int(row['id.resp_p']), row['proto'])
            quality, _ = leg_b_quality(row)
            services = parse_services(row.get('service'))
            if quality and not services.confirmed and identity not in pairs_seen:
                port_key = (row['proto'], int(row['id.resp_p']))
                counts = reference[port_key]
                share = counts['labeled'] / counts['total'] if counts['total'] else 0.0
                if counts['total'] >= reference_floor and share >= labeled_share:
                    seeds.setdefault(identity, {'timestamp': timestamp, 'identity': identity, 'reference': counts['total'], 'labeled_share': share})
        surfaced.extend(seeds.values())
        for row in batch:
            identity = (row['id.orig_h'], row['id.resp_h'], int(row['id.resp_p']), row['proto'])
            pairs_seen.add(identity)
            quality, _ = leg_b_quality(row)
            if quality:
                key = (row['proto'], int(row['id.resp_p']))
                reference[key]['total'] += 1
                reference[key]['labeled'] += bool(parse_services(row.get('service')).confirmed)
    return surfaced

def fixture_row(ts, src, service):
    row = {'ts': ts, 'id.orig_h': src, 'id.resp_h': '198.51.100.10', 'id.resp_p': 443, 'proto': 'tcp', 'service': service, 'history': 'ShDd', 'missed_bytes': 0, 'orig_bytes': 32, 'resp_bytes': 32, 'orig_ip_bytes': 60, 'resp_ip_bytes': 60, 'conn_state': 'SF'}
    row.update({OBSERVATION_FLAGS[field]: True for field in OBSERVED_FIELDS})
    return row

fixture = [fixture_row(1.0, '192.0.2.1', 'ssl'), fixture_row(1.0, '192.0.2.2', 'ssl'), fixture_row(2.0, '192.0.2.3', None), fixture_row(2.0, '192.0.2.3', None)]
seeds = leg_b_walk(fixture, reference_floor=2, labeled_share=1.0)
assert len(seeds) == 1 and seeds[0]['timestamp'] == 2.0
header_only = fixture_row(3.0, '192.0.2.4', None)
header_only['orig_bytes'] = 0
assert header_only['orig_ip_bytes'] > 0 and leg_b_quality(header_only)[0] is False
missing_service_schema = fixture_row(3.0, '192.0.2.5', None)
missing_service_schema['_source_has_service'] = False
assert leg_b_quality(missing_service_schema)[0] is False
flipped_unlabeled = fixture_row(3.0, '192.0.2.6', None)
flipped_unlabeled['history'] = 'ShDd^'
assert leg_b_quality(flipped_unlabeled)[0] is False
flipped_labeled = fixture_row(3.0, '192.0.2.7', 'ssl')
flipped_labeled['history'] = 'ShDd^'
assert leg_b_quality(flipped_labeled)[0] is True
seen_before_clean = fixture_row(0.5, '192.0.2.3', None)
seen_before_clean['conn_state'] = 'S0'
assert leg_b_walk([*fixture[:2], seen_before_clean, *fixture[2:]], reference_floor=2, labeled_share=1.0) == []
with tempfile.TemporaryDirectory() as directory:
    scale_fixture_path = Path(directory) / 'conn.ndjson'
    base_ts = datetime(2026, 5, 18, tzinfo=timezone.utc).timestamp()
    scale_rows = [fixture_row(base_ts, '192.0.2.1', 'ssl'), fixture_row(base_ts, '192.0.2.2', 'ssl'), fixture_row(base_ts + 1, '192.0.2.3', None), fixture_row(base_ts + 1, '192.0.2.3', None)]
    scale_fixture_path.write_text('\n'.join(json.dumps(row) for row in scale_rows) + '\n', encoding='utf-8')
    scale_result = scan_leg_b_week((scale_fixture_path,), DEVELOPMENT_WEEKS[0])
    assert scale_result['new_eligible_seed_events'] == 1
    assert all(item['surfaced_seed_events'] == 0 for item in scale_result['grid'])
print('leg B timestamp batching, distinct populations, payload-byte/schema gates and bounded scale harness passed')

In [ ]:
LEG_A_RESULT = EXPECTATION_TABLE.parent / 'leg-a-dev-4x7d-2026-05-18_2026-06-08_2026-06-29_2026-07-20-v8.2.1-r3-structural-role.json'
LEG_A_SHARE_BAR = 0.001
LEG_A_DAYS_BAR = 2

leg_a_result = json.loads(LEG_A_RESULT.read_text(encoding='utf-8'))
assert tuple(leg_a_result['development_weeks']) == DEVELOPMENT_WEEKS
mismatches = [record for record in leg_a_result['records'] if record['verdict'] == 'mismatch']
share_only = [record for record in mismatches if record['share'] < LEG_A_SHARE_BAR]
rare = [record for record in share_only if record['days_present'] < LEG_A_DAYS_BAR]
routine = [record for record in mismatches if record not in rare]
assert leg_a_result['evaluation_losses']['role_inverted'] == 2
assert len(mismatches) == 14 and len(share_only) == 11 and len(rare) == 10 and len(routine) == 4
assert max(record['share'] for record in rare) == 0.00030727329864869583
assert min(record['share'] for record in routine) == 9.959767457873872e-05
assert min(record['share'] for record in routine if record['share'] >= LEG_A_SHARE_BAR) == 0.0053074478857502005
assert {record['days_present'] for record in rare} == {1}

RARE_EXPLANATIONS = {
    ('http', 443, 'tcp'): 'cleartext HTTP confirmed on a TLS-standard port; unresolved pending local owner mapping',
    ('http', 1337, 'tcp'): 'candidate application-specific HTTP listener; unresolved pending local owner mapping',
    ('http', 1400, 'tcp'): 'candidate application-specific HTTP listener; unresolved pending local owner mapping',
    ('http', 2086, 'tcp'): 'candidate application-specific HTTP listener; unresolved pending local owner mapping',
    ('http', 2095, 'tcp'): 'candidate application-specific HTTP listener; unresolved pending local owner mapping',
    ('http', 2710, 'tcp'): 'candidate application-specific HTTP listener; unresolved pending local owner mapping',
    ('http', 6969, 'tcp'): 'candidate application-specific HTTP listener; unresolved pending local owner mapping',
    ('http', 7676, 'tcp'): 'candidate application-specific HTTP listener; unresolved pending local owner mapping',
    ('http', 9197, 'tcp'): 'candidate application-specific HTTP listener; unresolved pending local owner mapping',
    ('http', 11450, 'tcp'): 'candidate application-specific HTTP listener; unresolved pending local owner mapping',
}
rare_keys = {(record['mismatched_services'][0], record['port'], record['proto']) for record in rare}
assert rare_keys == set(RARE_EXPLANATIONS)
print(f'leg A bars: share < {LEG_A_SHARE_BAR:.3%} and days < {LEG_A_DAYS_BAR}; rare={len(rare)}, routine={len(routine)}')
print('share gap:', max(record['share'] for record in rare), 'to', min(record['share'] for record in routine if record['share'] >= LEG_A_SHARE_BAR))

In [ ]:
import numpy as np

LEG_C_CONN_STATES = ('S0', 'S1', 'SF', 'REJ', 'S2', 'S3', 'RSTO', 'RSTR', 'RSTOS0', 'RSTRH', 'SH', 'SHR', 'OTH')
LEG_C_HISTORY_LETTERS = tuple('SsHhAaDdFfRrIiQqCcTtWwGg^x')
LEG_C_REQUIRED_FLAGS = ('service', 'history', 'orig_pkts', 'resp_pkts', 'orig_ip_bytes', 'resp_ip_bytes')
LEG_C_NUMERIC_NAMES = ('log1p_duration', 'log1p_orig_ip_bytes', 'log1p_resp_ip_bytes', 'log1p_orig_pkts', 'log1p_resp_pkts', 'orig_bytes_per_packet', 'resp_bytes_per_packet', 'orig_byte_share')
LEG_C_BINARY_NAMES = tuple(f'conn_state={state}' for state in (*LEG_C_CONN_STATES, 'other')) + tuple(f'history={letter}' for letter in LEG_C_HISTORY_LETTERS) + ('orig_packetless', 'resp_packetless')

def leg_c_features(row: dict) -> dict:
    missing_flags = [field for field in LEG_C_REQUIRED_FLAGS if row.get(f'_source_has_{field}') is not True]
    if missing_flags:
        return {'status': 'not_evaluable', 'reason': 'required-input-not-supplied', 'missing': missing_flags}
    if row.get('conn_state') is None or row.get('history') is None:
        return {'status': 'not_evaluable', 'reason': 'required-value-unset'}
    services = parse_services(row.get('service'))
    if not services.confirmed:
        return {'status': 'not_evaluable', 'reason': 'no-confirmed-service'}
    if not isinstance(row.get('proto'), str) or not row['proto']:
        return {'status': 'not_evaluable', 'reason': 'transport-unset-or-unknown'}
    raw = {name: row.get(name) for name in ('duration', 'orig_pkts', 'resp_pkts', 'orig_ip_bytes', 'resp_ip_bytes')}
    try:
        values = {name: float(value) for name, value in raw.items()}
    except (TypeError, ValueError):
        return {'status': 'not_evaluable', 'reason': 'non-finite-or-negative'}
    if any(not math.isfinite(value) or value < 0 for value in values.values()):
        return {'status': 'not_evaluable', 'reason': 'non-finite-or-negative'}
    op, rp = values['orig_pkts'], values['resp_pkts']
    ob, rb = values['orig_ip_bytes'], values['resp_ip_bytes']
    numeric = np.asarray((math.log1p(values['duration']), math.log1p(ob), math.log1p(rb), math.log1p(op), math.log1p(rp), ob / op if op else 0.0, rb / rp if rp else 0.0, ob / (ob + rb) if ob + rb else 0.5), dtype=np.float64)
    state = str(row.get('conn_state') or 'OTH')
    state = state if state in LEG_C_CONN_STATES else 'other'
    history = str(row.get('history') or '')
    binary = np.asarray(tuple(float(state == candidate) for candidate in (*LEG_C_CONN_STATES, 'other')) + tuple(float(letter in history) for letter in LEG_C_HISTORY_LETTERS) + (float(op == 0), float(rp == 0)), dtype=np.float64)
    return {'status': 'eligible', 'class_key': (services.confirmed, str(row['proto'])), 'numeric': numeric, 'binary': binary, 'unknown_history_letters': len(set(history) - set(LEG_C_HISTORY_LETTERS))}

def fit_leg_c_scaler(numeric_rows: np.ndarray, fallback_cap: float) -> dict:
    if numeric_rows.ndim != 2 or numeric_rows.shape[1] != len(LEG_C_NUMERIC_NAMES):
        raise ValueError('numeric feature shape mismatch')
    if numeric_rows.shape[0] == 0:
        raise ValueError('numeric feature rows are empty')
    if not math.isfinite(fallback_cap) or fallback_cap <= 0:
        raise ValueError('fallback_cap must be finite and positive')
    centers = np.median(numeric_rows, axis=0)
    q25, q75 = np.percentile(numeric_rows, (25, 75), axis=0)
    iqr = q75 - q25
    constants = np.ptp(numeric_rows, axis=0) == 0
    scales = iqr.copy()
    fallback_kind = ['iqr'] * numeric_rows.shape[1]
    for column in np.flatnonzero((iqr == 0) & ~constants):
        mad = float(np.median(np.abs(numeric_rows[:, column] - centers[column])))
        span = float(np.ptp(numeric_rows[:, column]))
        raw_scale, kind = (mad, 'mad') if mad > 0 else ((span, 'range') if span > 0 else (1.0, 'unit'))
        scales[column] = min(raw_scale, fallback_cap)
        fallback_kind[column] = kind
    scales[constants] = 1.0
    return {'centers': centers, 'scales': scales, 'retained': ~constants, 'constant_names': tuple(name for name, drop in zip(LEG_C_NUMERIC_NAMES, constants, strict=True) if drop), 'scale_kind': tuple(fallback_kind)}

def apply_leg_c_scaler(numeric_rows: np.ndarray, binary_rows: np.ndarray, scaler: dict) -> np.ndarray:
    retained_numeric = (numeric_rows[:, scaler['retained']] - scaler['centers'][scaler['retained']]) / scaler['scales'][scaler['retained']]
    return np.concatenate((retained_numeric, binary_rows), axis=1)

def leg_c_constant_blind_spot(numeric_rows: np.ndarray, scaler: dict) -> np.ndarray:
    dropped = ~scaler['retained']
    return np.any(numeric_rows[:, dropped] != scaler['centers'][dropped], axis=1) if dropped.any() else np.zeros(numeric_rows.shape[0], dtype=bool)

def leg_c_content_key(features: dict) -> tuple:
    if features.get('status') != 'eligible':
        raise ValueError('only eligible rows have a fit-order key')
    return tuple(float(value).hex() for value in features['numeric']) + tuple(int(value) for value in features['binary'])

def seeded_fit_rows(features: list[dict], cap: int, seed: int) -> list[dict]:
    if isinstance(cap, bool) or cap <= 0:
        raise ValueError('fit-row cap must be a positive integer')
    ordered = sorted(features, key=leg_c_content_key)
    permutation = np.random.Generator(np.random.PCG64(seed)).permutation(len(ordered))
    return [ordered[int(index)] for index in permutation[:cap]]

leg_c_base = {'service': 'ssl', 'proto': 'tcp', 'duration': 2.0, 'orig_pkts': 2, 'resp_pkts': 1, 'orig_ip_bytes': 200, 'resp_ip_bytes': 100, 'conn_state': 'SF', 'history': 'ShADadFf', **{f'_source_has_{field}': True for field in LEG_C_REQUIRED_FLAGS}}
leg_c_feature_fixture = leg_c_features(leg_c_base)
assert leg_c_feature_fixture['status'] == 'eligible' and leg_c_feature_fixture['unknown_history_letters'] == 0
assert leg_c_features({**leg_c_base, '_source_has_history': False})['reason'] == 'required-input-not-supplied'
assert leg_c_features({**leg_c_base, 'history': None})['reason'] == 'required-value-unset'
assert leg_c_features({**leg_c_base, 'service': None})['reason'] == 'no-confirmed-service'
assert leg_c_features({**leg_c_base, 'duration': float('nan')})['reason'] == 'non-finite-or-negative'
assert leg_c_features({**leg_c_base, 'orig_pkts': 0, 'orig_ip_bytes': 0})['numeric'][-3] == 0.0
assert leg_c_features({**leg_c_base, 'orig_ip_bytes': 0, 'resp_ip_bytes': 0})['numeric'][-1] == 0.5
scale_fixture = np.asarray([[1, 1, 0, 0, 0, 0, 5, 0], [1, 2, 0, 0, 0, 0, 5, 1], [1, 3, 0, 0, 0, 0, 5, 1], [1, 4, 0, 0, 0, 0, 5, 1], [1, 5, 1, 0, 0, 0, 5, 1]], dtype=np.float64)
scale = fit_leg_c_scaler(scale_fixture, fallback_cap=10.0)
assert set(scale['constant_names']) == {'log1p_duration', 'log1p_orig_pkts', 'log1p_resp_pkts', 'orig_bytes_per_packet', 'resp_bytes_per_packet'}
assert scale['scale_kind'][2] == 'range'
scaled = apply_leg_c_scaler(scale_fixture, np.zeros((5, len(LEG_C_BINARY_NAMES))), scale)
assert scaled.shape == (5, 3 + len(LEG_C_BINARY_NAMES)) and np.isfinite(scaled).all()
assert leg_c_constant_blind_spot(scale_fixture, scale).sum() == 0
assert leg_c_constant_blind_spot(np.asarray([[2, 1, 0, 0, 0, 0, 5, 0]], dtype=np.float64), scale).tolist() == [True]
order_fixture = [leg_c_features({**leg_c_base, 'duration': duration}) for duration in (5, 1, 3, 2, 4)]
selected = seeded_fit_rows(order_fixture, cap=3, seed=20260909)
assert [leg_c_content_key(item) for item in selected] == [leg_c_content_key(item) for item in seeded_fit_rows(list(reversed(order_fixture)), cap=3, seed=20260909)]
assert fit_leg_c_class(order_fixture, class_floor=6, surviving_floor=1, fit_cap=3, fallback_cap=10, topology=(4,), seed=1, early_stopping=True, validation_fraction=0.2, max_iter=10, batch_size=2)['status'] == 'below_class_floor'
bar_fixture = robust_score_bar(np.asarray([*range(10), 100], dtype=np.float64), population_floor=10, multiplier=6, cliff_gap=10)
assert bar_fixture['status'] == 'ready' and bar_fixture['threshold'] < 100 and bar_fixture['observed_cliff_gap'] >= 10
assert robust_score_bar(np.asarray(range(10), dtype=np.float64), population_floor=10, multiplier=6, cliff_gap=10)['status'] == 'no_tail'
pair_fixture = aggregate_pair_scores([('a',), ('a',), ('b',)], np.asarray([1.0, 10.0, 11.0]), threshold=5.0, share_floor=0.5)
assert [(row['pair_key'], row['share']) for row in pair_fixture] == [(('a',), 0.5), (('b',), 1.0)]
fold_fixture = [{'pair_key': (f'src-{index}', 'dst', 443, 'tcp', ('ssl',))} for index in range(4)]
assert folded_pair_finding_count(fold_fixture) == 1 and folded_pair_finding_count(fold_fixture[:3]) == 3
print(f'leg C row eligibility, stable fit sampling, constant-axis blind spots, and robust scaling mechanics passed: {len(LEG_C_NUMERIC_NAMES)} numeric candidates, {len(LEG_C_BINARY_NAMES)} unit-scale binaries')
print(f'leg C MLP abstention and bounded-batch scoring mechanics loaded under scikit-learn {SKLEARN_VERSION}; calibration settings remain unset')

In [ ]:
def scan_leg_c_inventory(paths: Iterable[Path]) -> dict:
    counts = Counter()
    classes = Counter()
    unknown_history = Counter()
    for path in paths:
        for row in iter_zeek_ndjson(path):
            counts['input_rows'] += 1
            features = leg_c_features(row)
            if features['status'] != 'eligible':
                counts[f"not_evaluable_{features['reason']}"] += 1
                continue
            counts['eligible_rows'] += 1
            classes[features['class_key']] += 1
            if features['unknown_history_letters']:
                counts['rows_with_unknown_history_letters'] += 1
                for letter in set(str(row.get('history') or '')) - set(LEG_C_HISTORY_LETTERS):
                    unknown_history[letter] += 1
    class_records = [
        {'services': list(services), 'proto': proto, 'rows': rows}
        for (services, proto), rows in sorted(classes.items())
    ]
    return {'schema_version': 1, 'development_weeks': DEVELOPMENT_WEEKS, 'sklearn_version': SKLEARN_VERSION, 'counts': dict(counts), 'classes': class_records, 'unknown_history_letters': dict(sorted(unknown_history.items()))}

def write_leg_c_inventory(result: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(result, sort_keys=True, indent=2) + '\n', encoding='utf-8')

LEG_C_DEVELOPMENT_INVENTORY = EXPECTATION_TABLE.parent / 'leg-c-dev-inventory-4x7d-2026-05-18_2026-06-08_2026-06-29_2026-07-20-v1.json'
# Director-owned scale run:
# write_leg_c_inventory(scan_leg_c_inventory(DEVELOPMENT_CONN_FILES), LEG_C_DEVELOPMENT_INVENTORY)
print('leg C privacy-clean development inventory harness loaded; no scan executed')

In [ ]:
def _leg_c_binary_mask(binary: np.ndarray) -> int:
    return sum(int(value) << index for index, value in enumerate(binary))

def _leg_c_binary_from_mask(mask: int) -> np.ndarray:
    return np.asarray(tuple(float(bool(mask & (1 << index))) for index in range(len(LEG_C_BINARY_NAMES))), dtype=np.float64)

def external_leg_c_fit_samples(rows: Iterable[dict], *, fit_cap: int, seed: int) -> dict:
    if isinstance(fit_cap, bool) or fit_cap <= 0:
        raise ValueError('fit_cap must be a positive integer')
    numeric_columns = ', '.join(f'n{index} REAL NOT NULL' for index in range(len(LEG_C_NUMERIC_NAMES)))
    order_columns = ', '.join(f'n{index}' for index in range(len(LEG_C_NUMERIC_NAMES))) + ', bmask'
    with tempfile.TemporaryDirectory(prefix='sigwood-protocol-legc-sample-') as directory:
        database = sqlite3.connect(Path(directory) / 'fit-order.sqlite3')
        database.execute('PRAGMA journal_mode=OFF')
        database.execute('PRAGMA synchronous=OFF')
        database.execute(f'CREATE TABLE eligible (class_key TEXT NOT NULL, {numeric_columns}, bmask INTEGER NOT NULL)')
        counts = Counter()
        batch = []
        placeholders = ','.join('?' for _ in range(2 + len(LEG_C_NUMERIC_NAMES)))
        for row in rows:
            features = leg_c_features(row)
            if features['status'] != 'eligible':
                continue
            class_text = json.dumps((features['class_key'][0], features['class_key'][1]), separators=(',', ':'))
            counts[class_text] += 1
            batch.append((class_text, *map(float, features['numeric']), _leg_c_binary_mask(features['binary'])))
            if len(batch) == 10000:
                database.executemany(f'INSERT INTO eligible VALUES ({placeholders})', batch)
                batch.clear()
        if batch:
            database.executemany(f'INSERT INTO eligible VALUES ({placeholders})', batch)
        database.execute('CREATE TABLE wanted (class_key TEXT NOT NULL, ordinal INTEGER NOT NULL, PRIMARY KEY (class_key, ordinal))')
        rng = np.random.Generator(np.random.PCG64(seed))
        wanted = []
        for class_text, count in sorted(counts.items()):
            ordinals = rng.choice(count, size=min(fit_cap, count), replace=False)
            wanted.extend((class_text, int(ordinal)) for ordinal in ordinals)
        database.executemany('INSERT INTO wanted VALUES (?, ?)', wanted)
        query = f'''WITH numbered AS (
            SELECT class_key, {order_columns}, row_number() OVER (PARTITION BY class_key ORDER BY {order_columns}) - 1 AS ordinal
            FROM eligible
        )
        SELECT numbered.class_key, {', '.join(f'numbered.n{index}' for index in range(len(LEG_C_NUMERIC_NAMES)))}, numbered.bmask
        FROM numbered JOIN wanted USING (class_key, ordinal)
        ORDER BY numbered.class_key, numbered.ordinal'''
        samples = defaultdict(list)
        for record in database.execute(query):
            class_text, *values, mask = record
            services, proto = json.loads(class_text)
            samples[(tuple(services), proto)].append({'status': 'eligible', 'class_key': (tuple(services), proto), 'numeric': np.asarray(values, dtype=np.float64), 'binary': _leg_c_binary_from_mask(mask), 'unknown_history_letters': 0})
        database.close()
    return dict(samples)

def development_leg_c_rows(paths: Iterable[Path]):
    for path in paths:
        yield from iter_zeek_ndjson(path)

def write_leg_c_fit_samples(samples: dict, path: Path) -> None:
    class_keys = sorted(samples)
    arrays = {'schema_version': np.asarray([1], dtype=np.int64), 'class_keys': np.asarray([json.dumps((services, proto), separators=(',', ':')) for services, proto in class_keys])}
    for index, class_key in enumerate(class_keys):
        arrays[f'numeric_{index}'] = np.stack([row['numeric'] for row in samples[class_key]])
        arrays[f'binary_{index}'] = np.stack([row['binary'] for row in samples[class_key]]).astype(np.uint8)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, **arrays)

def read_leg_c_fit_samples(path: Path) -> dict:
    result = {}
    with np.load(path, allow_pickle=False) as arrays:
        if arrays['schema_version'].tolist() != [1]:
            raise ValueError('unsupported fit-sample schema')
        for index, class_text in enumerate(arrays['class_keys'].tolist()):
            services, proto = json.loads(class_text)
            class_key = (tuple(services), proto)
            result[class_key] = [
                {'status': 'eligible', 'class_key': class_key, 'numeric': numeric, 'binary': binary.astype(np.float64), 'unknown_history_letters': 0}
                for numeric, binary in zip(arrays[f'numeric_{index}'], arrays[f'binary_{index}'], strict=True)
            ]
    return result

LEG_C_SAMPLE_SEED = 20260909
LEG_C_CANDIDATE_MAX_FIT_CAP = 50000
LEG_C_DEVELOPMENT_FIT_SAMPLES = EXPECTATION_TABLE.parent / 'leg-c-dev-fit-samples-cap50000-seed20260909-v1.npz'
LEG_C_DEVELOPMENT_MODEL_GRID = EXPECTATION_TABLE.parent / 'leg-c-dev-model-grid-cap50000-seed20260909-iter500-v2.json'
# Director-owned scale run:
# samples = external_leg_c_fit_samples(development_leg_c_rows(DEVELOPMENT_CONN_FILES), fit_cap=LEG_C_CANDIDATE_MAX_FIT_CAP, seed=LEG_C_SAMPLE_SEED)
# write_leg_c_fit_samples(samples, LEG_C_DEVELOPMENT_FIT_SAMPLES)
# write_leg_c_model_grid(run_leg_c_model_grid(read_leg_c_fit_samples(LEG_C_DEVELOPMENT_FIT_SAMPLES)), LEG_C_DEVELOPMENT_MODEL_GRID)

external_fixture_rows = [{**leg_c_base, 'duration': float(value)} for value in range(1, 13)]
external_sample_a = external_leg_c_fit_samples(external_fixture_rows, fit_cap=5, seed=20260909)
external_sample_b = external_leg_c_fit_samples(reversed(external_fixture_rows), fit_cap=5, seed=20260909)
assert [leg_c_content_key(row) for row in external_sample_a[(('ssl',), 'tcp')]] == [leg_c_content_key(row) for row in external_sample_b[(('ssl',), 'tcp')]]
with tempfile.TemporaryDirectory() as directory:
    fit_sample_path = Path(directory) / 'samples.npz'
    write_leg_c_fit_samples(external_sample_a, fit_sample_path)
    roundtrip = read_leg_c_fit_samples(fit_sample_path)
    assert [leg_c_content_key(row) for row in roundtrip[(('ssl',), 'tcp')]] == [leg_c_content_key(row) for row in external_sample_a[(('ssl',), 'tcp')]]
print('leg C external full-content-order fit sampler passed its source-order negative control')

In [ ]:
LEG_C_ADVANCED_TOPOLOGY = (32, 16, 32)
LEG_C_ROBUST_MULTIPLIERS = (6.0, 10.0, 20.0)
LEG_C_CLIFF_GAPS = (1.0, 2.0, 4.0)
LEG_C_PAIR_SHARES = (0.1, 0.25, 0.5)
LEG_C_FROZEN_MULTIPLIER = 10.0
LEG_C_FROZEN_CLIFF_GAP = 2.0
LEG_C_FROZEN_PAIR_SHARE = 0.5
LEG_C_REDESIGN_MULTIPLIER = 20.0
LEG_C_REDESIGN_CLIFF_GAP = 0.0
LEG_C_REDESIGN_PAIR_SHARE = 0.5
LEG_C_DEVELOPMENT_POSITIVE = Path.home() / '.sigwood/bench/external/c2-beacon-pcaps/zeek/tunneled-c2-beaconing/conn.log'
LEG_C_DEVELOPMENT_POSITIVE_RESULT = EXPECTATION_TABLE.parent / 'leg-c-dev-positive-sliver-ligolo-medium-v2-folded.json'

def _leg_c_pair_key(row: dict, class_key: tuple) -> tuple | None:
    identity = _valid_identity(row)
    if identity is None:
        return None
    _ts, src, dst, port, proto = identity
    return src, dst, port, proto, class_key[0]

def run_leg_c_positive(path: Path, *, scenario: str, target_src: str, target_dst: str, target_port: int, multipliers: tuple[float, ...] = LEG_C_ROBUST_MULTIPLIERS, cliff_gaps: tuple[float, ...] = LEG_C_CLIFF_GAPS, pair_shares: tuple[float, ...] = LEG_C_PAIR_SHARES) -> dict:
    rows = read_zeek_tsv(path)
    by_class = defaultdict(list)
    counts = Counter(input_rows=len(rows))
    target_rows = 0
    for row in rows:
        features = leg_c_features(row)
        if features['status'] != 'eligible':
            counts[f"not_evaluable_{features['reason']}"] += 1
            continue
        pair_key = _leg_c_pair_key(row, features['class_key'])
        if pair_key is None:
            counts['invalid_identity'] += 1
            continue
        counts['eligible_rows'] += 1
        is_target = pair_key[0] == target_src and pair_key[1] == target_dst and pair_key[2] == target_port
        target_rows += int(is_target)
        by_class[features['class_key']].append((features, pair_key, is_target))
    grid = {}
    for method in ('autoencoder', 'robust-z'):
        for multiplier in multipliers:
            for cliff_gap in cliff_gaps:
                for pair_share in pair_shares:
                    grid[(method, multiplier, cliff_gap, pair_share)] = {'method': method, 'multiplier': multiplier, 'cliff_gap': cliff_gap, 'pair_share': pair_share, 'pair_findings': 0, 'target_surfaced': False, 'bar_statuses': Counter()}
    class_statuses = []
    for class_key, class_rows in sorted(by_class.items()):
        features = [item[0] for item in class_rows]
        pair_keys = [item[1] for item in class_rows]
        target_flags = [item[2] for item in class_rows]
        fitted = fit_leg_c_class(features, class_floor=LEG_C_CANDIDATE_CLASS_FLOOR, surviving_floor=LEG_C_CANDIDATE_SURVIVING_FLOOR, fit_cap=LEG_C_CANDIDATE_MAX_FIT_CAP, fallback_cap=LEG_C_CANDIDATE_FALLBACK_CAP, topology=LEG_C_ADVANCED_TOPOLOGY, seed=LEG_C_SAMPLE_SEED, early_stopping=True, validation_fraction=LEG_C_CANDIDATE_VALIDATION_FRACTION, max_iter=LEG_C_CANDIDATE_MAX_ITER, batch_size=LEG_C_CANDIDATE_BATCH_SIZE)
        class_statuses.append({'services': list(class_key[0]), 'proto': class_key[1], 'rows': len(features), 'status': fitted['status'], 'iterations': fitted.get('iterations')})
        if fitted['status'] != 'modeled':
            continue
        method_scores = {'autoencoder': score_leg_c_class(features, fitted, score_batch_size=4096)['errors'], 'robust-z': score_robust_z_baseline(features, fitted['scaler'])['scores']}
        for method, scores in method_scores.items():
            for multiplier in multipliers:
                for cliff_gap in cliff_gaps:
                    bar = robust_score_bar(scores, population_floor=LEG_C_CANDIDATE_CLASS_FLOOR, multiplier=multiplier, cliff_gap=cliff_gap)
                    for pair_share in pair_shares:
                        record = grid[(method, multiplier, cliff_gap, pair_share)]
                        record['bar_statuses'][bar['status']] += 1
                        if bar['status'] != 'ready':
                            continue
                        findings = aggregate_pair_scores(pair_keys, scores, threshold=bar['threshold'], share_floor=pair_share)
                        record['pair_findings'] += folded_pair_finding_count(findings)
                        finding_keys = {item['pair_key'] for item in findings}
                        record['target_surfaced'] |= any(flag and key in finding_keys for key, flag in zip(pair_keys, target_flags, strict=True))
    records = []
    for record in grid.values():
        records.append({**record, 'bar_statuses': dict(record['bar_statuses'])})
    parameters = {'multipliers': list(multipliers), 'cliff_gaps': list(cliff_gaps), 'pair_shares': list(pair_shares)}
    return {'schema_version': 1, 'scenario': scenario, 'parameters': parameters, 'counts': dict(counts), 'target_rows': target_rows, 'class_statuses': class_statuses, 'grid': records}

def write_leg_c_positive(result: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(result, sort_keys=True, indent=2) + '\n', encoding='utf-8')

def run_leg_c_frozen_positive(path: Path, *, scenario: str, target_src: str, target_dst: str, target_port: int) -> dict:
    return run_leg_c_positive(path, scenario=scenario, target_src=target_src, target_dst=target_dst, target_port=target_port, multipliers=(LEG_C_FROZEN_MULTIPLIER,), cliff_gaps=(LEG_C_FROZEN_CLIFF_GAP,), pair_shares=(LEG_C_FROZEN_PAIR_SHARE,))

def run_leg_c_redesign_positive(path: Path, *, scenario: str, target_src: str, target_dst: str, target_port: int) -> dict:
    return run_leg_c_positive(path, scenario=scenario, target_src=target_src, target_dst=target_dst, target_port=target_port, multipliers=(LEG_C_REDESIGN_MULTIPLIER,), cliff_gaps=(LEG_C_REDESIGN_CLIFF_GAP,), pair_shares=(LEG_C_REDESIGN_PAIR_SHARE,))

# Director-owned development positive run; target identity comes from the private corpus manifest and never enters the artifact:
# write_leg_c_positive(run_leg_c_positive(LEG_C_DEVELOPMENT_POSITIVE, scenario='sliver-ligolo-tunnel', target_src=..., target_dst=..., target_port=11601), LEG_C_DEVELOPMENT_POSITIVE_RESULT)
print('leg C development-positive grid harness loaded; held-back positive remains unopened')

In [ ]:
def fit_selected_leg_c_models(samples: dict) -> dict:
    return {
        class_key: fit_leg_c_class(rows, class_floor=LEG_C_CANDIDATE_CLASS_FLOOR, surviving_floor=LEG_C_CANDIDATE_SURVIVING_FLOOR, fit_cap=LEG_C_CANDIDATE_MAX_FIT_CAP, fallback_cap=LEG_C_CANDIDATE_FALLBACK_CAP, topology=LEG_C_ADVANCED_TOPOLOGY, seed=LEG_C_SAMPLE_SEED, early_stopping=True, validation_fraction=LEG_C_CANDIDATE_VALIDATION_FRACTION, max_iter=LEG_C_CANDIDATE_MAX_ITER, batch_size=LEG_C_CANDIDATE_BATCH_SIZE)
        for class_key, rows in sorted(samples.items())
    }

def _score_leg_c_batch(buffer: list, fitted: dict) -> tuple:
    features = [item[0] for item in buffer]
    pairs = [item[1] for item in buffer]
    auto = score_leg_c_class(features, fitted, score_batch_size=len(features))['errors']
    baseline = score_robust_z_baseline(features, fitted['scaler'])['scores']
    return pairs, auto, baseline

def iter_leg_c_scored_batches(paths: Iterable[Path], models: dict, *, batch_size: int = 4096):
    buffers = defaultdict(list)
    for path in paths:
        for row in iter_zeek_ndjson(path):
            features = leg_c_features(row)
            if features['status'] != 'eligible':
                continue
            fitted = models.get(features['class_key'])
            if not fitted or fitted['status'] != 'modeled':
                continue
            pair_key = _leg_c_pair_key(row, features['class_key'])
            if pair_key is None:
                continue
            buffer = buffers[features['class_key']]
            buffer.append((features, pair_key))
            if len(buffer) == batch_size:
                yield features['class_key'], _score_leg_c_batch(buffer, fitted)
                buffer.clear()
    for class_key, buffer in sorted(buffers.items()):
        if buffer:
            yield class_key, _score_leg_c_batch(buffer, models[class_key])

def collect_leg_c_score_populations(paths: Iterable[Path], models: dict) -> dict:
    chunks = defaultdict(lambda: {'autoencoder': [], 'robust-z': []})
    for class_key, (_pairs, auto, baseline) in iter_leg_c_scored_batches(paths, models):
        chunks[class_key]['autoencoder'].append(auto)
        chunks[class_key]['robust-z'].append(baseline)
    return {class_key: {method: np.concatenate(values) for method, values in methods.items()} for class_key, methods in chunks.items()}

def leg_c_candidate_bars(populations: dict, *, multipliers: tuple[float, ...] = LEG_C_ROBUST_MULTIPLIERS, cliff_gaps: tuple[float, ...] = LEG_C_CLIFF_GAPS) -> dict:
    bars = {}
    for class_key, methods in populations.items():
        for method, scores in methods.items():
            for multiplier in multipliers:
                for cliff_gap in cliff_gaps:
                    bars[(class_key, method, multiplier, cliff_gap)] = robust_score_bar(scores, population_floor=LEG_C_CANDIDATE_CLASS_FLOOR, multiplier=multiplier, cliff_gap=cliff_gap)
    return bars

def collect_leg_c_pair_aggregates(paths: Iterable[Path], models: dict, bars: dict, *, multipliers: tuple[float, ...] = LEG_C_ROBUST_MULTIPLIERS, cliff_gap: float = LEG_C_CLIFF_GAPS[0]) -> dict:
    aggregates = defaultdict(lambda: defaultdict(lambda: {'rows': 0, 'above': Counter()}))
    for class_key, (pairs, auto, baseline) in iter_leg_c_scored_batches(paths, models):
        for method, scores in (('autoencoder', auto), ('robust-z', baseline)):
            thresholds = {}
            for multiplier in multipliers:
                bar = bars[(class_key, method, multiplier, cliff_gap)]
                if 'threshold' in bar:
                    thresholds[multiplier] = bar['threshold']
            for pair_key, score in zip(pairs, scores, strict=True):
                if not math.isfinite(float(score)):
                    continue
                aggregate = aggregates[(class_key, method)][pair_key]
                aggregate['rows'] += 1
                for multiplier, threshold in thresholds.items():
                    aggregate['above'][multiplier] += int(score > threshold)
    return aggregates

def summarize_leg_c_benign_grid(aggregates: dict, bars: dict, *, multipliers: tuple[float, ...] = LEG_C_ROBUST_MULTIPLIERS, cliff_gaps: tuple[float, ...] = LEG_C_CLIFF_GAPS, pair_shares: tuple[float, ...] = LEG_C_PAIR_SHARES) -> list[dict]:
    records = []
    class_keys = sorted({key[0] for key in aggregates})
    for method in ('autoencoder', 'robust-z'):
        for multiplier in multipliers:
            for cliff_gap in cliff_gaps:
                for pair_share in pair_shares:
                    fold_groups = Counter()
                    bar_statuses = Counter()
                    raw_pairs = 0
                    for class_key in class_keys:
                        bar = bars[(class_key, method, multiplier, cliff_gap)]
                        bar_statuses[bar['status']] += 1
                        if bar['status'] != 'ready':
                            continue
                        for pair_key, aggregate in aggregates[(class_key, method)].items():
                            if aggregate['above'][multiplier] / aggregate['rows'] >= pair_share:
                                raw_pairs += 1
                                fold_groups[(pair_key[4], pair_key[2], pair_key[3])] += 1
                    folded = sum(1 if members >= 4 else members for members in fold_groups.values())
                    records.append({'method': method, 'multiplier': multiplier, 'cliff_gap': cliff_gap, 'pair_share': pair_share, 'raw_pair_findings': raw_pairs, 'pair_findings': folded, 'bar_statuses': dict(bar_statuses)})
    return records

def run_leg_c_benign_week(paths: Iterable[Path], week: str, *, multipliers: tuple[float, ...] = LEG_C_ROBUST_MULTIPLIERS, cliff_gaps: tuple[float, ...] = LEG_C_CLIFF_GAPS, pair_shares: tuple[float, ...] = LEG_C_PAIR_SHARES) -> dict:
    paths = tuple(paths)
    if not paths or any(path.parent.name not in _week_dates(week) for path in paths):
        raise ValueError('paths must belong only to the named week')
    started = time.perf_counter()
    samples = external_leg_c_fit_samples(development_leg_c_rows(paths), fit_cap=LEG_C_CANDIDATE_MAX_FIT_CAP, seed=LEG_C_SAMPLE_SEED)
    models = fit_selected_leg_c_models(samples)
    populations = collect_leg_c_score_populations(paths, models)
    bars = leg_c_candidate_bars(populations, multipliers=multipliers, cliff_gaps=cliff_gaps)
    aggregates = collect_leg_c_pair_aggregates(paths, models, bars, multipliers=multipliers, cliff_gap=cliff_gaps[0])
    model_statuses = [{'services': list(class_key[0]), 'proto': class_key[1], 'sample_rows': len(samples[class_key]), 'status': fitted['status'], 'iterations': fitted.get('iterations')} for class_key, fitted in sorted(models.items())]
    parameters = {'multipliers': list(multipliers), 'cliff_gaps': list(cliff_gaps), 'pair_shares': list(pair_shares)}
    return {'schema_version': 1, 'week': week, 'parameters': parameters, 'elapsed_seconds': time.perf_counter() - started, 'model_statuses': model_statuses, 'score_populations': [{'services': list(class_key[0]), 'proto': class_key[1], 'rows': len(methods['autoencoder'])} for class_key, methods in sorted(populations.items())], 'grid': summarize_leg_c_benign_grid(aggregates, bars, multipliers=multipliers, cliff_gaps=cliff_gaps, pair_shares=pair_shares)}

def write_leg_c_benign_week(result: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(result, sort_keys=True, indent=2) + '\n', encoding='utf-8')

def run_leg_c_frozen_benign_week(paths: Iterable[Path], week: str) -> dict:
    return run_leg_c_benign_week(paths, week, multipliers=(LEG_C_FROZEN_MULTIPLIER,), cliff_gaps=(LEG_C_FROZEN_CLIFF_GAP,), pair_shares=(LEG_C_FROZEN_PAIR_SHARE,))

def run_leg_c_redesign_benign_week(paths: Iterable[Path], week: str) -> dict:
    return run_leg_c_benign_week(paths, week, multipliers=(LEG_C_REDESIGN_MULTIPLIER,), cliff_gaps=(LEG_C_REDESIGN_CLIFF_GAP,), pair_shares=(LEG_C_REDESIGN_PAIR_SHARE,))

print('leg C bounded two-pass benign-week scoring harness loaded; no estate scan executed')

## Calibration boundary

The cells above freeze mechanics and the selected leg A and leg B thresholds. The leg C initial comparison ran only multiplier 10, cliff gap 2, and pair share 0.5 through the dedicated frozen wrappers and failed its positive-recall gate. The bounded redesign diagnostics removed the cliff at multipliers 10 and 20; both recovered the development positive but made autoencoder benign cost exceed robust-z, so no redesign was frozen. Leg B is admitted at its measured held-back rate. Both redesign-reserve weeks remain unopened, and leg C is a negative result pending the Author's product choice.